In [48]:
import os
import random
from gtts import gTTS
from pydub import AudioSegment

ffmpeg_folder = os.path.expanduser("~/ffmpeg-7.0.2-amd64-static")
os.environ["PATH"] += os.pathsep + ffmpeg_folder
AudioSegment.converter = os.path.join(ffmpeg_folder, "ffmpeg")
AudioSegment.ffprobe = os.path.join(ffmpeg_folder, "ffprobe")

cntScam = 1
cntNonScam = 1
out_mal = "dataset/malicious"
out_norm = "dataset/normal"

os.makedirs(out_mal, exist_ok=True)
os.makedirs(out_norm, exist_ok=True)

noise_file = "sounds/phone-call-background-noise.wav" 

print(f"Încerc să încarc zgomotul din: {noise_file}")
if not os.path.exists(noise_file):
    raise FileNotFoundError(f"Nu găsesc fișierul de zgomot: {noise_file}. Verifică folderul sounds!")

noise = AudioSegment.from_wav(noise_file) - 18 

input_file = "dataset/fraud_call.file"
if not os.path.exists(input_file):
     raise FileNotFoundError(f"Nu găsesc fișierul cu text: {input_file}")

with open(input_file, "r") as f:
    lines = f.readlines()
    print(f"Procesez {len(lines)} linii...")

    for i, line in enumerate(lines):
        line = line.strip()
        if not line: continue

        if "\t" in line:
            parts = line.split("\t", 1)
        else:
            parts = line.split(" ", 1)

        if len(parts) != 2:
            print(f"⚠️ Linia {i} ignorată (format incorect): {line[:20]}...")
            continue
            
        scam_type = parts[0].strip()
        text = parts[1].strip()

        if scam_type == "fraud":
            outfile = f"{out_mal}/fraud_call{cntScam}.wav"
            cntScam += 1
        elif scam_type == "normal":
            outfile = f"{out_norm}/fraud_call{cntNonScam}.wav"
            cntNonScam += 1
        else:
            continue

        try:

            temp_mp3 = "tmp_tts.mp3"
            tts = gTTS(text=text, lang="en")
            tts.save(temp_mp3)

            voice = AudioSegment.from_mp3(temp_mp3)
            
            # Uniformizăm (Mono, 22050Hz) ca să nu avem erori la mixaj
            voice = voice.set_channels(1).set_frame_rate(22050)
            noise = noise.set_channels(1).set_frame_rate(22050)

            if len(noise) < len(voice):
                looped_noise = noise * (len(voice) // len(noise) + 2)
            else:
                looped_noise = noise

            noise_trimmed = looped_noise[:len(voice)]
            mixed = voice.overlay(noise_trimmed)
            
            mixed.export(outfile, format="wav")

        except Exception as e:
            print(f"❌ Eroare la linia {i}: {e}")
            continue

# Curățenie
if os.path.exists("tmp_tts.mp3"):
    os.remove("tmp_tts.mp3")

print(f"✅ DONE. Generat: {cntScam-1} Fraude | {cntNonScam-1} Normale.")

Încerc să încarc zgomotul din: sounds/phone-call-background-noise.wav
Procesez 5932 linii...


KeyboardInterrupt: 

In [58]:

import IPython
IPython.display.Audio("tmp_tts.mp3")


In [ ]:
#Intreaba ce faci cu fisierele audio lungi